# MT-SICS Backend Hardware Validation

Run this notebook connected to a physical Mettler Toledo scale to validate
every implemented backend method. Each cell tests one method and logs the
raw MT-SICS command/response.

**Requirements:**
- Physical Mettler Toledo scale connected via USB-to-serial
- Scale powered on and warmed up (60 min)
- Weighing pan empty at start

---
## 1. Setup and Logging

In [1]:
import logging
from datetime import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO
from pylabrobot.scales import Scale
from pylabrobot.scales.mettler_toledo import MettlerToledoWXS205SDUBackend

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = f"./logs/hardware_validation/{timestamp}_validation.log"

pylabrobot.verbose(True, level=LOG_LEVEL_IO)
pylabrobot.setup_logger(log_dir=log_file, level=LOG_LEVEL_IO)

plr_logger = logging.getLogger("pylabrobot")
plr_logger.info("=== Hardware validation started ===")

2026-03-30 16:33:54,376 - pylabrobot - INFO - === Hardware validation started ===


In [2]:
# Update port for your system
backend = MettlerToledoWXS205SDUBackend(
  port="/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0"
)
scale = Scale(name="validation_scale", backend=backend, size_x=0, size_y=0, size_z=0)

await scale.setup()

2026-03-30 16:33:54,399 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0 (for VID=1027, PID=24577)
2026-03-30 16:33:54,407 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 16:33:54,408 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: @
2026-03-30 16:33:54,410 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 16:33:54,468 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 16:33:54,469 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 16:33:54,474 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I0
2026-03-30 16:33:54,477 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 16:33:54,519 - pyla

### Verify setup populated device identity

In [3]:
print(f"Device type:    {backend.device_type}")
print(f"Serial number:  {backend.serial_number}")
print(f"Capacity:       {backend.capacity} g")
print(
  f"Supported commands ({len(backend._supported_commands)}): {sorted(backend._supported_commands)}"
)

assert backend.device_type is not None
assert backend.serial_number is not None
assert backend.capacity > 0
assert "S" in backend._supported_commands
print("PASS: setup populated all identity fields")

Device type:    WXS205SDU WXA-Bridge
Serial number:  B207696838
Capacity:       220.009 g
Supported commands (62): ['@', 'C0', 'C1', 'C2', 'C3', 'COM', 'DAT', 'FCUT', 'FSET', 'I0', 'I1', 'I10', 'I11', 'I14', 'I15', 'I16', 'I2', 'I21', 'I22', 'I23', 'I24', 'I25', 'I26', 'I3', 'I4', 'I5', 'LST', 'M01', 'M02', 'M03', 'M17', 'M18', 'M19', 'M20', 'M21', 'M27', 'M28', 'M29', 'M31', 'M32', 'M33', 'M35', 'RDB', 'S', 'SI', 'SIR', 'SIS', 'SNR', 'SR', 'T', 'TA', 'TAC', 'TI', 'TIM', 'TST0', 'TST1', 'TST2', 'TST3', 'UPD', 'USTB', 'Z', 'ZI']
PASS: setup populated all identity fields


### Supported Python methods on this device

In [4]:
methods = backend.request_supported_methods()
print(f"{len(methods)} methods available on this device:")
for m in methods:
  print(f"  {m}")

65 methods available on this device:
  cancel_all
  clear_tare
  deserialize
  get_all_instances
  get_weight
  measure_temperature
  read_dynamic_weight
  read_stable_weight
  read_stable_weight_repeat_on_change
  read_weight
  read_weight_value_immediately
  request_adjustment_history
  request_adjustment_setting
  request_adjustment_weight
  request_assortment_type_revision
  request_auto_zero
  request_capacity
  request_date
  request_device_id
  request_device_info
  request_device_type
  request_environment_condition
  request_filter_cutoff
  request_firmware_version
  request_model_designation
  request_net_weight_with_status
  request_next_service_date
  request_operating_mode
  request_operating_mode_after_restart
  request_profact_day
  request_profact_temperature_criterion
  request_profact_time
  request_profact_time_criteria
  request_readability
  request_serial_number
  request_serial_parameters
  request_software_material_number
  request_stability_criteria
  request_s

### I0 - Discover all implemented commands

This tells us exactly which MT-SICS commands this specific device supports,
resolving whether SC, C, D are truly unsupported or just miscategorized by level.

In [5]:
from collections import defaultdict

# I0 is a multi-response command (B status for each command, A for the last)
responses = await backend.send_command("I0")

print(f"Total commands implemented: {len(responses)}")
print()

# Group by level
by_level = defaultdict(list)
for resp in responses:
  if len(resp.data) >= 2:
    level = resp.data[0]
    cmd_name = resp.data[1]
    by_level[level].append(cmd_name)

for level in sorted(by_level.keys()):
  cmds = sorted(by_level[level])
  print(f"Level {level} ({len(cmds)} commands): {', '.join(cmds)}")


2026-03-30 16:33:55,649 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I0
2026-03-30 16:33:55,652 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I0\r\n'
2026-03-30 16:33:55,684 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I0"\r\n'
2026-03-30 16:33:55,686 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I0 B 0 "I0"\r\n'
2026-03-30 16:33:55,700 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I1"\r\n'
2026-03-30 16:33:55,703 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I0 B 0 "I1"\r\n'
2026-03-30 16:33:55,716 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I0 B 0 "I2"\r\n'
2026-03-30 16:33:55,721 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I0 B 0 "I2"\r\n'
2026-03-30 16:33:55,727 - pylabrobot.io.serial - IO - [/dev/s

Total commands implemented: 62

Level 0 (12 commands): @, I0, I1, I2, I3, I4, I5, S, SI, SIR, Z, ZI
Level 1 (5 commands): SR, T, TA, TAC, TI
Level 2 (40 commands): C0, C1, C2, C3, COM, DAT, I10, I11, I14, I15, I16, I21, I22, I23, I24, I25, I26, M01, M02, M03, M17, M18, M19, M20, M21, M27, M28, M29, M31, M32, M33, M35, SIS, SNR, TIM, TST0, TST1, TST2, TST3, UPD
Level 3 (5 commands): FCUT, FSET, LST, RDB, USTB


### Device identity and diagnostics (Batch 1)

These commands complete the device identity picture beyond I2/I4.

In [6]:
# I3 - Firmware version (Level 0, always available)
fw_version = await backend.request_firmware_version()
print(f"Firmware version: {fw_version}")

# I5 - Software material number (Level 0, always available)
sw_material = await backend.request_software_material_number()
print(f"Software material number: {sw_material}")

# I10 - Device ID (user-assignable name)
if "I10" in backend._supported_commands:
  device_id = await backend.request_device_id()
  print(f"Device ID: '{device_id}'")
else:
  print("SKIP: I10 not supported")

# I11 - Model designation
if "I11" in backend._supported_commands:
  model = await backend.request_model_designation()
  print(f"Model designation: {model}")
else:
  print("SKIP: I11 not supported")

# I15 - Uptime (minutes since start/restart)
if "I15" in backend._supported_commands:
  uptime_min = await backend.request_uptime_minutes()
  print(f"Uptime: {uptime_min} minutes ({uptime_min / 60:.1f} hours)")
else:
  print("SKIP: I15 not supported")

# I16 - Next service date
if "I16" in backend._supported_commands:
  service_date = await backend.request_next_service_date()
  print(f"Next service date: {service_date}")
else:
  print("SKIP: I16 not supported")

# I21 - Assortment type revision
if "I21" in backend._supported_commands:
  revision = await backend.request_assortment_type_revision()
  print(f"Assortment type revision: {revision}")
else:
  print("SKIP: I21 not supported")

# I26 - Operating mode after restart
if "I26" in backend._supported_commands:
  i26_resp = await backend.request_operating_mode_after_restart()
  for resp in i26_resp:
    print(f"I26: {resp.command} {resp.status} {" ".join(resp.data)}")
else:
  print("SKIP: I26 not supported")

# DAT - Date
if "DAT" in backend._supported_commands:
  date = await backend.request_date()
  print(f"Device date: {date}")
else:
  print("SKIP: DAT not supported")

# TIM - Time
if "TIM" in backend._supported_commands:
  time_str = await backend.request_time()
  print(f"Device time: {time_str}")
else:
  print("SKIP: TIM not supported")

# I14 - Comprehensive device info (query each category separately)
if "I14" in backend._supported_commands:
  category_names = {
    0: "Instrument configuration",
    1: "Instrument descriptions",
    2: "SW identification numbers",
    3: "SW versions",
    4: "Serial numbers",
    5: "TDNR numbers",
  }
  print("\nDevice info (I14):")
  for cat, name in category_names.items():
    try:
      info = await backend.request_device_info(category=cat)
      print(f"  Category {cat} ({name}):")
      for resp in info:
        print(f"    {resp.command} {resp.status} {" ".join(resp.data)}")
    except Exception as e:
      print(f"  Category {cat} ({name}): {e}")
else:
  print("SKIP: I14 not supported")

print("\nPASS: Batch 1 device identity commands completed")


2026-03-30 16:33:56,576 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I3
2026-03-30 16:33:56,579 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I3\r\n'
2026-03-30 16:33:56,630 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I3 A "1.10 18.6.4.1361.772"\r\n'
2026-03-30 16:33:56,636 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I3 A "1.10 18.6.4.1361.772"\r\n'
2026-03-30 16:33:56,639 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I5
2026-03-30 16:33:56,643 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I5\r\n'
2026-03-30 16:33:56,675 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I5 A "11671158C"\r\n'
2026-03-30 16:33:56,676 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I5 A "11671158C"\r\n'
2026-03-30 16:33:56,680 - pylabrobot - IO - [Mettler

Firmware version: 1.10 18.6.4.1361.772
Software material number: 11671158C
Device ID: ''


2026-03-30 16:33:56,771 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I11 A "WXS205SDU"\r\n'
2026-03-30 16:33:56,772 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I11 A "WXS205SDU"\r\n'
2026-03-30 16:33:56,773 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I15
2026-03-30 16:33:56,775 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I15\r\n'
2026-03-30 16:33:56,807 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I15 A 424\r\n'
2026-03-30 16:33:56,808 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I15 A 424\r\n'
2026-03-30 16:33:56,810 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I16
2026-03-30 16:33:56,818 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I16\r\n'


Model designation: WXS205SDU
Uptime: 424 minutes (7.1 hours)


2026-03-30 16:33:57,298 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I16 A 16 03 2013\r\n'
2026-03-30 16:33:57,299 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I16 A 16 03 2013\r\n'
2026-03-30 16:33:57,301 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I21
2026-03-30 16:33:57,303 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I21\r\n'
2026-03-30 16:33:57,330 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I21 A "5"\r\n'
2026-03-30 16:33:57,332 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I21 A "5"\r\n'
2026-03-30 16:33:57,335 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I26
2026-03-30 16:33:57,339 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I26\r\n'
2026-03-30 16:33:57,363 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC

Next service date: 16 03 2013
Assortment type revision: 5
I26: I26 A 0
Device date: 04
Device time: 23

Device info (I14):


2026-03-30 16:33:57,507 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 0 1 "Bridge"\r\n'
2026-03-30 16:33:57,512 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I14 A 0 1 "Bridge"\r\n'
2026-03-30 16:33:57,515 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I14 1
2026-03-30 16:33:57,517 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 1\r\n'
2026-03-30 16:33:57,554 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 1 1 "WXS205SDU"\r\n'
2026-03-30 16:33:57,554 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I14 A 1 1 "WXS205SDU"\r\n'
2026-03-30 16:33:57,555 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I14 2
2026-03-30 16:33:57,558 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I14 2\r\n'
2026-03-30 16:33:57,603 - pylabrobot.io.serial -

  Category 0 (Instrument configuration):
    I14 A 0 1 Bridge
  Category 1 (Instrument descriptions):
    I14 A 1 1 WXS205SDU
  Category 2 (SW identification numbers):
    I14 A 2 1 11671158C
  Category 3 (SW versions):
    I14 A 3 1 1.10
  Category 4 (Serial numbers):
    I14 A 4 1 B207696838


2026-03-30 16:33:57,746 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I14 A 5 1 "18.6.4.1361.772"\r\n'
2026-03-30 16:33:57,747 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I14 A 5 1 "18.6.4.1361.772"\r\n'


  Category 5 (TDNR numbers):
    I14 A 5 1 18.6.4.1361.772

PASS: Batch 1 device identity commands completed


---
## 2. Core Protocol Validation

These Level 0/1 commands have been validated across multiple hardware runs.
This cell consolidates them into a single pass/fail check.

In [7]:
# @ - Cancel (returns serial number)
sn = await backend.reset()
assert sn == backend.serial_number, f"reset() returned {sn}, expected {backend.serial_number}"

# I4 - Serial number
sn2 = await backend.request_serial_number()
assert sn2 == sn

# I2 - Device type and capacity
dt = await backend.request_device_type()
assert len(dt) > 0
cap = await backend.request_capacity()
assert cap > 0

# S - Stable weight
w_stable = await backend.read_stable_weight()
assert isinstance(w_stable, float)

# SI - Weight immediately
w_imm = await backend.read_weight_value_immediately()
assert isinstance(w_imm, float)

# Z - Zero (stable) then read
await scale.zero(timeout="stable")
w_after_zero = await backend.read_weight_value_immediately()
assert abs(w_after_zero) < 0.001, f"Expected ~0 after zero, got {w_after_zero}"

# ZI - Zero immediately then read
await scale.zero(timeout=0)
w_after_zi = await backend.read_weight_value_immediately()
assert abs(w_after_zi) < 0.01

# TA - Request tare weight
tare = await scale.request_tare_weight()
assert isinstance(tare, float)

# TAC - Clear tare
await backend.clear_tare()
tare_after = await scale.request_tare_weight()
assert abs(tare_after) < 0.001

print("PASS: all core Level 0/1 commands validated")

2026-03-30 16:33:57,762 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 16:33:57,766 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: @
2026-03-30 16:33:57,770 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 16:33:57,827 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 16:33:57,828 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 16:33:57,829 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I4
2026-03-30 16:33:57,833 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I4\r\n'
2026-03-30 16:33:57,874 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 16:33:57,875 - pylabrobot - IO - [Mettler T

PASS: all core Level 0/1 commands validated


---
## 3. Level 1 Commands (Elementary)

**Place a known weight on the scale for tare tests.**

### T - Tare (stable)

In [8]:
input("Place a container on the scale and press Enter...")
await scale.zero(timeout="stable")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Tare weight: {tare} g")
assert tare > 0, f"Expected positive tare, got {tare}"
print("PASS: tare stores container weight")

Place a container on the scale and press Enter... 


2026-03-30 16:34:04,612 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: Z
2026-03-30 16:34:04,614 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 16:34:04,988 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 16:34:04,988 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'Z A\r\n'
2026-03-30 16:34:04,989 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: T
2026-03-30 16:34:04,992 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 16:34:05,388 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 16:34:05,388 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 16:34:05,391 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: TA
2026-03-30 16:34:05,

Tare weight: 0.0 g


AssertionError: Expected positive tare, got 0.0

### TA - Request tare weight

In [9]:
tare = await scale.request_tare_weight()
print(f"Tare weight from memory: {tare} g")
assert isinstance(tare, float)
print("PASS: tare weight readable from memory")

2026-03-30 16:34:12,398 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: TA
2026-03-30 16:34:12,401 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 16:34:12,469 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 16:34:12,472 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'TA A    0.00000 g\r\n'


Tare weight from memory: 0.0 g
PASS: tare weight readable from memory


### TAC - Clear tare

In [10]:
await backend.clear_tare()
tare_after_clear = await scale.request_tare_weight()
print(f"Tare after clear: {tare_after_clear} g")
assert abs(tare_after_clear) < 0.001, f"Expected ~0 after clear, got {tare_after_clear}"
print("PASS: clear tare resets tare to 0")

2026-03-30 16:34:13,336 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: TAC
2026-03-30 16:34:13,339 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TAC\r\n'
2026-03-30 16:34:13,365 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TAC A\r\n'
2026-03-30 16:34:13,367 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'TAC A\r\n'
2026-03-30 16:34:13,368 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: TA
2026-03-30 16:34:13,374 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 16:34:13,445 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 16:34:13,448 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'TA A    0.00000 g\r\n'


Tare after clear: 0.0 g
PASS: clear tare resets tare to 0


---
## 4. Level 2 Commands (Extended)

### M21 - Set host unit to grams

In [11]:
if "M21" in backend._supported_commands:
  await backend.set_host_unit_grams()
  print("PASS: host unit set to grams")
else:
  print("SKIP: M21 not supported on this device")

2026-03-30 16:34:16,488 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M21 0 0
2026-03-30 16:34:16,489 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M21 0 0\r\n'
2026-03-30 16:34:16,529 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M21 A\r\n'
2026-03-30 16:34:16,530 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M21 A\r\n'


PASS: host unit set to grams


### I50 - Remaining weighing range (multi-response)

In [12]:
if "I50" in backend._supported_commands:
  remaining = await backend.request_remaining_weighing_range()
  print(f"Remaining weighing range: {remaining} g")
  assert remaining > 0
  assert remaining <= backend.capacity
  print("PASS: remaining range is positive and within capacity")
else:
  print("SKIP: I50 not supported on this device")

SKIP: I50 not supported on this device


### M28 - Temperature sensor

In [13]:
if "M28" in backend._supported_commands:
  temp = await backend.measure_temperature()
  print(f"Scale temperature: {temp} C")
  assert 5 < temp < 50, f"Temperature {temp} C outside reasonable range"
  print("PASS: measure_temperature works - I0 correctly predicted M28 support")
else:
  print("SKIP: M28 not supported on this device")

2026-03-30 16:34:18,754 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M28
2026-03-30 16:34:18,756 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M28\r\n'
2026-03-30 16:34:18,800 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 B 1 19.5\r\n'
2026-03-30 16:34:18,804 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M28 B 1 19.5\r\n'
2026-03-30 16:34:18,808 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M28 A 2 18.9\r\n'
2026-03-30 16:34:18,813 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M28 A 2 18.9\r\n'


Scale temperature: 19.5 C
PASS: measure_temperature works - I0 correctly predicted M28 support


### Batch 2 - Configuration queries (read-only)

These commands read device configuration without modifying anything.

In [14]:
# M01 - Weighing mode
if "M01" in backend._supported_commands:
  mode = await backend.request_weighing_mode()
  mode_names = {
    0: "Normal/Universal",
    1: "Dosing",
    2: "Sensor",
    3: "Check weighing",
    6: "Raw/No filter",
  }
  print(f"Weighing mode: {mode} ({mode_names.get(mode, 'unknown')})")
else:
  print("SKIP: M01 not supported")

# M02 - Environment condition
if "M02" in backend._supported_commands:
  env = await backend.request_environment_condition()
  env_names = {
    0: "Very stable",
    1: "Stable",
    2: "Standard",
    3: "Unstable",
    4: "Very unstable",
    5: "Automatic",
  }
  print(f"Environment condition: {env} ({env_names.get(env, 'unknown')})")
else:
  print("SKIP: M02 not supported")

# M03 - Auto zero
if "M03" in backend._supported_commands:
  auto_zero = await backend.request_auto_zero()
  print(f"Auto zero: {auto_zero} ({'on' if auto_zero else 'off'})")
else:
  print("SKIP: M03 not supported")

# M17 - ProFACT time criteria
if "M17" in backend._supported_commands:
  resp = await backend.request_profact_time_criteria()
  print(f"ProFACT time criteria: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M17 not supported")

# M18 - ProFACT temperature criterion
if "M18" in backend._supported_commands:
  resp = await backend.request_profact_temperature_criterion()
  print(f"ProFACT temperature criterion: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M18 not supported")

# M19 - Adjustment weight
if "M19" in backend._supported_commands:
  resp = await backend.request_adjustment_weight()
  print(f"Adjustment weight: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M19 not supported")

# M20 - Test weight
if "M20" in backend._supported_commands:
  resp = await backend.request_test_weight()
  print(f"Test weight: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M20 not supported")

# M27 - Adjustment history
if "M27" in backend._supported_commands:
  history = await backend.request_adjustment_history()
  print(f"\nAdjustment history ({len(history)} entries):")
  for resp in history:
    print(f"  {resp.command} {resp.status} {" ".join(resp.data)}")
else:
  print("SKIP: M27 not supported")

# M29 - Weighing value release
if "M29" in backend._supported_commands:
  resp = await backend.request_weighing_value_release()
  print(f"Weighing value release: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M29 not supported")

# M31 - Operating mode
if "M31" in backend._supported_commands:
  resp = await backend.request_operating_mode()
  print(f"Operating mode: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M31 not supported")

# M32 - ProFACT time
if "M32" in backend._supported_commands:
  resp = await backend.request_profact_time()
  print(f"ProFACT time: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M32 not supported")

# M33 - ProFACT day
if "M33" in backend._supported_commands:
  resp = await backend.request_profact_day()
  print(f"ProFACT day: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M33 not supported")

# M35 - Zeroing mode
if "M35" in backend._supported_commands:
  resp = await backend.request_zeroing_mode()
  print(f"Zeroing mode: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: M35 not supported")

# UPD - Update rate
if "UPD" in backend._supported_commands:
  rate = await backend.request_update_rate()
  print(f"\nUpdate rate: {rate} values/s")
else:
  print("SKIP: UPD not supported")

# SIS - Net weight with status
if "SIS" in backend._supported_commands:
  sis_resp = await backend.request_net_weight_with_status()
  print(f"Net weight with status: {sis_resp.command} {sis_resp.status} {" ".join(sis_resp.data)}")
else:
  print("SKIP: SIS not supported")

# C0 - Adjustment setting
if "C0" in backend._supported_commands:
  resp = await backend.request_adjustment_setting()
  print(f"Adjustment setting: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: C0 not supported")

# COM - Serial parameters
if "COM" in backend._supported_commands:
  resp = await backend.request_serial_parameters()
  print(f"Serial parameters: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: COM not supported")

# FCUT - Filter cut-off frequency
if "FCUT" in backend._supported_commands:
  resp = await backend.request_filter_cutoff()
  print(f"Filter cut-off: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: FCUT not supported")

# LST - Current user settings
if "LST" in backend._supported_commands:
  lst_responses = await backend.request_user_settings()
  print(f"\nCurrent user settings ({len(lst_responses)} lines):")
  for resp in lst_responses[:10]:
    print(f"  {resp.command} {resp.status} {" ".join(resp.data)}")
  if len(lst_responses) > 10:
    print(f"  ... ({len(lst_responses)} lines total)")
else:
  print("SKIP: LST not supported")

# RDB - Readability
if "RDB" in backend._supported_commands:
  resp = await backend.request_readability()
  print(f"Readability: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: RDB not supported")

# USTB - Stability criteria
if "USTB" in backend._supported_commands:
  resp = await backend.request_stability_criteria()
  print(f"Stability criteria: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: USTB not supported")

# TST0 - Test settings
if "TST0" in backend._supported_commands:
  resp = await backend.request_test_settings()
  print(f"Test settings: {resp[0].status} {" ".join(resp[0].data)}")
else:
  print("SKIP: TST0 not supported")

print("\nPASS: All read-only configuration queries completed")


2026-03-30 16:34:20,135 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M01
2026-03-30 16:34:20,137 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M01\r\n'
2026-03-30 16:34:20,158 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M01 A 0\r\n'
2026-03-30 16:34:20,158 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M01 A 0\r\n'
2026-03-30 16:34:20,161 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M02
2026-03-30 16:34:20,162 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M02\r\n'
2026-03-30 16:34:20,191 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M02 A 2\r\n'
2026-03-30 16:34:20,192 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M02 A 2\r\n'
2026-03-30 16:34:20,193 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M03
2026-03-30 16:34:20,1

Weighing mode: 0 (Normal/Universal)
Environment condition: 2 (Standard)
Auto zero: 0 (off)
ProFACT time criteria: A 00 00 00 0
ProFACT temperature criterion: A 1
Adjustment weight: A 10.00000 g


2026-03-30 16:34:20,398 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M20 A 200.00000 g\r\n'
2026-03-30 16:34:20,399 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M20 A 200.00000 g\r\n'
2026-03-30 16:34:20,400 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M27
2026-03-30 16:34:20,403 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M27\r\n'
2026-03-30 16:34:20,461 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 1 13 05 2020 17 17 0 ""\r\n'
2026-03-30 16:34:20,462 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M27 B 1 13 05 2020 17 17 0 ""\r\n'
2026-03-30 16:34:20,495 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 2 02 12 2019 20 02 0 ""\r\n'
2026-03-30 16:34:20,497 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M27 B 2 02

Test weight: A 200.00000 g


2026-03-30 16:34:20,621 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 6 14 03 2019 20 18 0 ""\r\n'
2026-03-30 16:34:20,623 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M27 B 6 14 03 2019 20 18 0 ""\r\n'
2026-03-30 16:34:20,653 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 7 17 10 2018 17 38 0 ""\r\n'
2026-03-30 16:34:20,654 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M27 B 7 17 10 2018 17 38 0 ""\r\n'
2026-03-30 16:34:20,685 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 8 17 10 2018 17 37 0 ""\r\n'
2026-03-30 16:34:20,686 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M27 B 8 17 10 2018 17 37 0 ""\r\n'
2026-03-30 16:34:20,717 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M27 B 9 17 10 2018 17 35 0 ""\r\n'
2026-03

...

Zeroing mode: A 0

Update rate: 10.173 values/s
Net weight with status: SIS A 0 0.00000 0 5 1 0 0
Adjustment setting: A 0 0 
Serial parameters: A 0 6 3 1


2026-03-30 16:34:22,540 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'FCUT A 0.000\r\n'
2026-03-30 16:34:22,541 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'FCUT A 0.000\r\n'
2026-03-30 16:34:22,543 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: LST
2026-03-30 16:34:22,545 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'LST\r\n'
2026-03-30 16:34:22,588 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B C0 0 0 ""\r\n'
2026-03-30 16:34:22,589 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B C0 0 0 ""\r\n'
2026-03-30 16:34:22,604 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B FCUT 0.000\r\n'
2026-03-30 16:34:22,605 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B FCUT 0.000\r\n'
2026-03-30 16:34:22,619 - pylabrobot.io

Filter cut-off: A 0.000


2026-03-30 16:34:22,747 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M20 200.00000 g\r\n'
2026-03-30 16:34:22,748 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B M20 200.00000 g\r\n'
2026-03-30 16:34:22,767 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M21 0 0\r\n'
2026-03-30 16:34:22,768 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B M21 0 0\r\n'
2026-03-30 16:34:22,779 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M29 1\r\n'
2026-03-30 16:34:22,782 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B M29 1\r\n'
2026-03-30 16:34:22,796 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'LST B M31 0\r\n'
2026-03-30 16:34:22,798 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'LST B M31 0\r\n'
2026


Current user settings (24 lines):
  LST B C0 0 0 
  LST B FCUT 0.000
  LST B I10 
  LST B M01 0
  LST B M02 2
  LST B M03 0
  LST B M17 00 00 00 0
  LST B M18 1
  LST B M19 10.00000 g
  LST B M20 200.00000 g
  ... (24 lines total)
Readability: A 5
Stability criteria: B 0 3.600 1.100
Test settings: A 0 

PASS: All read-only configuration queries completed


### SIS unit code verification (exploratory)

Tests whether field index 4 in the SIS response is the unit code by temporarily
changing the host unit via M21, reading SIS, then restoring grams.

**WARNING:** Temporarily writes a non-gram unit to device memory via M21.
If the cell fails mid-execution, run `await backend.set_host_unit_grams()`
manually to restore grams. setup() also restores grams on next connection.

In [15]:
# Read SIS in grams (current state)
sis_grams = await backend.request_net_weight_with_status()
print(f"SIS in grams:      {sis_grams.data}")

# Temporarily set unit to milligrams (M21 1 0 = mg host unit)
try:
  await backend.send_command("M21 1 0")
  sis_mg = await backend.request_net_weight_with_status()
  print(f"SIS in milligrams: {sis_mg.data}")

  # Compare the unit code field (data[2] per SIS spec)
  if len(sis_grams.data) > 2 and len(sis_mg.data) > 2:
    print(f"\nUnit code in grams mode: {sis_grams.data[2]}")
    print(f"Unit code in mg mode:    {sis_mg.data[2]}")
    if sis_grams.data[2] != sis_mg.data[2]:
      print("PASS: unit code changes with M21 - confirms data[2] is unit identifier")
    else:
      print("INFO: unit code did not change - field may not be the unit identifier")
  else:
    print("INFO: SIS response too short to compare unit codes")

finally:
  # Always restore grams
  await backend.set_host_unit_grams()
  print("\nRestored host unit to grams")


2026-03-30 16:34:35,694 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: SIS
2026-03-30 16:34:35,697 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SIS\r\n'
2026-03-30 16:34:35,743 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'SIS A 0 "0.00000" 0 5 1 0 0\r\n'
2026-03-30 16:34:35,744 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'SIS A 0 "0.00000" 0 5 1 0 0\r\n'
2026-03-30 16:34:35,745 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: M21 1 0
2026-03-30 16:34:35,748 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'M21 1 0\r\n'
2026-03-30 16:34:35,779 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'M21 L\r\n'
2026-03-30 16:34:35,781 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'M21 L\r\n'
2026-03-30 16:34:35,783 - pylabrobot - IO - [Mettler Toledo Sc

SIS in grams:      ['0', '0.00000', '0', '5', '1', '0', '0']

Restored host unit to grams


MettlerToledoError: Command understood but not executable: (incorrect parameter).

---
## 5. Frontend Integration

Test the Scale frontend methods that delegate to the backend.

In [16]:
# Zero via frontend
await scale.zero(timeout="stable")
w = await scale.read_weight(timeout=0)
print(f"Frontend zero + read: {w} g")
assert abs(w) < 0.001

# Tare via frontend
input("Place a container on the scale and press Enter...")
await scale.tare(timeout="stable")
tare = await scale.request_tare_weight()
print(f"Frontend tare weight: {tare} g")
assert tare > 0

# Read via frontend
w = await scale.read_weight(timeout="stable")
print(f"Frontend read (should be ~0 with container tared): {w} g")
assert abs(w) < 0.1

print("PASS: all frontend methods delegate correctly")

2026-03-30 16:34:46,597 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: Z
2026-03-30 16:34:46,599 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'Z\r\n'
2026-03-30 16:34:46,854 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'Z A\r\n'
2026-03-30 16:34:46,856 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'Z A\r\n'
2026-03-30 16:34:46,857 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: SI
2026-03-30 16:34:46,860 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'SI\r\n'
2026-03-30 16:34:46,903 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 16:34:46,904 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'S S    0.00000 g\r\n'


Frontend zero + read: 0.0 g


Place a container on the scale and press Enter... 


2026-03-30 16:34:48,056 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: T
2026-03-30 16:34:48,060 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'T\r\n'
2026-03-30 16:34:48,453 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'T S    0.00000 g\r\n'
2026-03-30 16:34:48,454 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'T S    0.00000 g\r\n'
2026-03-30 16:34:48,455 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: TA
2026-03-30 16:34:48,456 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'TA\r\n'
2026-03-30 16:34:48,533 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'TA A    0.00000 g\r\n'
2026-03-30 16:34:48,533 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'TA A    0.00000 g\r\n'


Frontend tare weight: 0.0 g


AssertionError: 

### I22-I25 - Undocumented information commands

These commands appear in the I0 discovery but are not documented in the MT-SICS spec.
All I-commands are read-only queries, so these are safe to probe.

In [17]:
for cmd in ["I22", "I23", "I24", "I25"]:
  try:
    responses = await backend.send_command(cmd)
    for r in responses:
      print(f"{cmd}: status={r.status} data={r.data}")
  except Exception as e:
    print(f"{cmd}: {type(e).__name__}: {e}")
  print()


2026-03-30 16:34:53,926 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: I22
2026-03-30 16:34:53,932 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'I22\r\n'
2026-03-30 16:34:54,319 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I22 B 0 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,320 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I22 B 0 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,400 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I22 B 1 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,401 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I22 B 1 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,480 - pylabrobot.io.serial - IO - [/dev/serial

I22: status=B data=['0', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I22: status=B data=['1', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I22: status=B data=['2', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I22: status=B data=['3', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I22: status=A data=['4', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']



2026-03-30 16:34:54,879 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I23 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,881 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I23 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,944 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I23 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:54,956 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I23 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,025 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I23 A 4 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,028 - pylabrobot - IO - [Mettler Toledo Scale] Received respo

I23: status=B data=['0', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I23: status=B data=['1', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I23: status=B data=['2', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I23: status=B data=['3', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I23: status=A data=['4', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']



2026-03-30 16:34:55,279 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I24 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,280 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I24 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,360 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I24 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,360 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I24 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,439 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I24 A 4 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,440 - pylabrobot - IO - [Mettler Toledo Scale] Received respo

I24: status=B data=['0', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I24: status=B data=['1', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I24: status=B data=['2', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I24: status=B data=['3', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I24: status=A data=['4', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']



2026-03-30 16:34:55,695 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I25 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,696 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I25 B 2 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,759 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I25 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,760 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I25 B 3 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,839 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I25 A 4 0.00000000E+00 0.00000000E+00 0.00000000E+00 0.00000000E+00 ""\r\n'
2026-03-30 16:34:55,839 - pylabrobot - IO - [Mettler Toledo Scale] Received respo

I25: status=B data=['0', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I25: status=B data=['1', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I25: status=B data=['2', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I25: status=B data=['3', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']
I25: status=A data=['4', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '0.00000000E+00', '']



---
## 6. Performance

In [18]:
import time

import numpy as np

# Stable read latency
times_stable = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout="stable")
  t1 = time.monotonic_ns()
  times_stable.append((t1 - t0) / 1e6)

# Immediate read latency
times_immediate = []
for _ in range(10):
  t0 = time.monotonic_ns()
  await scale.read_weight(timeout=0)
  t1 = time.monotonic_ns()
  times_immediate.append((t1 - t0) / 1e6)

print(f"Stable read:    {np.mean(times_stable):.2f} +/- {np.std(times_stable):.2f} ms")
print(f"Immediate read: {np.mean(times_immediate):.2f} +/- {np.std(times_immediate):.2f} ms")

2026-03-30 16:35:10,903 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: S
2026-03-30 16:35:10,904 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 16:35:10,948 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 16:35:10,949 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 16:35:10,950 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: S
2026-03-30 16:35:10,954 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'S\r\n'
2026-03-30 16:35:11,043 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'S S    0.00000 g\r\n'
2026-03-30 16:35:11,044 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'S S    0.00000 g\r\n'
2026-03-30 16:35:11,045 - pylabrobot - IO - [Mettler Toledo Scale] Sent comman

Stable read:    92.43 +/- 15.88 ms
Immediate read: 44.97 +/- 6.03 ms


---
## 7. Teardown

In [19]:
plr_logger.info("=== Hardware validation ended ===")
await scale.stop()
print(f"Log file: {log_file}")

2026-03-30 16:35:13,600 - pylabrobot - INFO - === Hardware validation ended ===
2026-03-30 16:35:13,602 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] reset_input_buffer
2026-03-30 16:35:13,603 - pylabrobot - IO - [Mettler Toledo Scale] Sent command: @
2026-03-30 16:35:13,604 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] write b'@\r\n'
2026-03-30 16:35:13,665 - pylabrobot.io.serial - IO - [/dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0] readline b'I4 A "B207696838"\r\n'
2026-03-30 16:35:13,667 - pylabrobot - IO - [Mettler Toledo Scale] Received response: b'I4 A "B207696838"\r\n'
2026-03-30 16:35:13,668 - pylabrobot - INFO - [Mettler Toledo Scale] Disconnected from /dev/serial/by-id/usb-FTDI_UC232R_FT9NOXXH-if00-port0


Log file: ./logs/hardware_validation/2026-03-30_16-33-54_validation.log
